In [7]:
from glob import glob
import pandas as pd
import os
import soundfile as sf
from tqdm import tqdm
from multiprocess import Pool
from scipy.io import wavfile
import itertools
import io
import numpy as np
import json
import re
import zipfile
import librosa
from pathlib import Path

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

In [2]:
from huggingface_hub import snapshot_download

snapshot_download(repo_id="ai4bharat/indicvoices_r", 
                  repo_type="dataset", local_dir="./indicvoices_r")

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Fetching 2345 files:  11%|█▏        | 266/2345 [01:59<25:41,  1.35it/s]

  2025-09-26T08:36:46.859910Z  WARN  Status Code: 504. Retrying..., request_id: ""
    at /home/runner/work/xet-core/xet-core/cas_client/src/http_client.rs:227

  2025-09-26T08:36:46.859947Z  WARN  Retry attempt #0. Sleeping 2.425867763s before the next attempt
    at /root/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/reqwest-retry-0.7.0/src/middleware.rs:171



Fetching 2345 files:  24%|██▍       | 565/2345 [04:00<06:34,  4.52it/s]

  2025-09-26T08:38:50.065857Z  WARN  Status Code: 504. Retrying..., request_id: ""
    at /home/runner/work/xet-core/xet-core/cas_client/src/http_client.rs:227

  2025-09-26T08:38:50.065886Z  WARN  Retry attempt #0. Sleeping 2.518195845s before the next attempt
    at /root/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/reqwest-retry-0.7.0/src/middleware.rs:171

  2025-09-26T08:38:50.231763Z  WARN  Status Code: 504. Retrying..., request_id: ""
    at /home/runner/work/xet-core/xet-core/cas_client/src/http_client.rs:227

  2025-09-26T08:38:50.231796Z  WARN  Retry attempt #0. Sleeping 1.353348309s before the next attempt
    at /root/.cargo/registry/src/index.crates.io-1949cf8c6b5b557f/reqwest-retry-0.7.0/src/middleware.rs:171



Fetching 2345 files: 100%|██████████| 2345/2345 [13:36<00:00,  2.87it/s]


'/home/ubuntu/indicvoices_r'

In [16]:
files = glob('indicvoices_r/*/*.parquet')
files = [f for f in files if 'train' in f]
len(files)

2302

In [22]:
def loop(files):

    os.environ['OMP_NUM_THREADS'] = '1'
    os.environ['OPENBLAS_NUM_THREADS'] = '1'
    
    files, _ = files

    data = []
    for f in tqdm(files):
        base = '_'.join(f.split('/')[:2]) + '_audio'
        f_new = f.replace('/', '-').replace('.parquet', '')
        os.makedirs(base, exist_ok=True)
        df = pd.read_parquet(f)
        for i in range(len(df)):
            t = df['verbatim'].iloc[i].strip()
            if len(t) < 2:
                continue
            speaker = df['speaker_id'].iloc[i].strip()
            if len(speaker) < 2:
                continue
            audio_filename = f'{f_new}_{i}.mp3'
            audio_filename = os.path.join(base, audio_filename)
            b = df['audio'].iloc[i]['bytes']
            audio_np, sr = librosa.load(io.BytesIO(b), sr = 24000)
            if audio_np.ndim > 1:
                audio_np = audio_np.mean(axis=1)
            if audio_np.shape[0] < 10000:
                continue
            sf.write(audio_filename, audio_np, sr)
            
            data.append({
                'audio_filename': audio_filename,
                'text': t,
                'speaker': f"{base}_{speaker}"
            })
        
    return data

In [18]:
data = loop((files[:1], 0))
data

  0%|          | 0/291 [00:00<?, ?it/s]


[{'audio_filename': 'indicvoices_r_Assamese_audio/indicvoices_r-Assamese-train-00162-of-00246_0.mp3',
  'text': 'ছাত্ৰ ছাত্ৰী তাত কোচিং কৰিবলে আহে',
  'speaker': 'indicvoices_r_Assamese_audio_S4259022000377830'}]

In [23]:
data = multiprocessing(files, loop, cores = 30)

100%|██████████| 22/22 [09:29<00:00, 25.88s/it]


In [24]:
len(data)

655110

In [25]:
from datasets import Dataset

dataset = Dataset.from_list(data)
dataset[0]

{'audio_filename': 'indicvoices_r_Assamese_audio/indicvoices_r-Assamese-train-00162-of-00246_0.mp3',
 'text': 'ছাত্ৰ ছাত্ৰী তাত কোচিং কৰিবলে আহে',
 'speaker': 'indicvoices_r_Assamese_audio_S4259022000377830'}

In [26]:
dataset.push_to_hub('malaysia-ai/Multilingual-TTS', 'indicvoices_r')

Creating parquet from Arrow format: 100%|██████████| 3/3 [00:00<00:00,  6.60ba/s]
Processing Files (0 / 0): |          |  0.00B /  0.00B            
Processing Files (0 / 1):  91%|█████████ | 67.9MB / 74.6MB, 7.72MB/s  
Processing Files (0 / 1):  99%|█████████▉| 74.2MB / 74.6MB, 8.24MB/s  
Processing Files (1 / 1): 100%|██████████| 74.6MB / 74.6MB, 7.77MB/s  
Processing Files (1 / 1): 100%|██████████| 74.6MB / 74.6MB, 7.46MB/s  
New Data Upload: 100%|██████████| 74.6MB / 74.6MB, 7.46MB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:12<00:00, 12.32s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS/commit/3df2ba43087bc330b382f3b052bea8b4b6f0da60', commit_message='Upload dataset', commit_description='', oid='3df2ba43087bc330b382f3b052bea8b4b6f0da60', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Multilingual-TTS', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Multilingual-TTS'), pr_revision=None, pr_num=None)

In [27]:
audio_files = [d['audio_filename'] for d in data]

with open('indicvoices_r-audio.json', 'w') as fopen:
    json.dump(list(set(audio_files)), fopen)

In [39]:
# folders = glob('indicvoices_r_*_audio_neucodec')
# folders = [f for f in folders if '.zip' not in f]
# for f in folders:
#     print(f)
#     os.system(f'zip -rq {f}.zip {f}')

In [38]:
# from huggingface_hub import HfApi
# api = HfApi()

# for f in glob('indicvoices_r_*_audio_neucodec.zip'):
#     print(f)
#     api.upload_file(
#         path_or_fileobj=f,
#         path_in_repo=f,
#         repo_id="malaysia-ai/Multilingual-TTS",
#         repo_type="dataset",
#     )